In [5]:
import numpy as np
import pandas as pd
import pickle
import sys
import os
import torch
import torch.nn as nn

sys.path.append('..')
sys.path.append('../src')

from src.text_features import clean_text, decode_labels, TFIDFVectorizer, truncate_text
from src.ffnn import FFNN

In [6]:
df_subm = pd.read_csv('subm1.csv', sep=";")

# sem truncação — o vectorizer foi treinado em textos completos
texts = [clean_text(t) for t in df_subm['Text'].tolist()]

with open('../vectorizers/vectorizer_dnn.pkl', 'rb') as f:
    vec = pickle.load(f)

X_subm = vec.transform(texts)
X_subm_dense = X_subm.toarray() if not isinstance(X_subm, np.ndarray) else X_subm
X_tensor = torch.tensor(X_subm_dense, dtype=torch.float32)

print(f"Dados transformados. (shape: {X_tensor.shape})")

Dados transformados. (shape: torch.Size([150, 20000]))


In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_tensor = X_tensor.to(device)

input_dim = X_tensor.shape[1]
model_dnn = FFNN(input_dim=input_dim, 
                 n_classes=5, 
                 topology=[256, 128], 
                 dropout=0.3).to(device)

model_dnn.load_state_dict(torch.load('../models/model_dnn.pt', map_location=device))

model_dnn.eval()

with torch.no_grad():
    outputs = model_dnn(X_tensor)
    preds_idx = torch.argmax(outputs, dim=1).cpu().numpy()